# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [1]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [ ]:
# atom() is provided in sudoku_solver.py and imported above.


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [ ]:
# Implement build_general_kb() and build_definite_kb() in sudoku_solver.py.
# Rerun the import cell near the top of this notebook after making changes.


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [ ]:
# Attempt (a): try solving the puzzle using pl_resolution / tt_entails on the
# general KB. Left commented out because it is expected to take an
# impractically long time (pl_resolution) or be outright infeasible
# (tt_entails) on the full grid.
#
# Uncomment a few lines at a time and give each at most ~30 seconds; use
# Kernel > Interrupt if it hasn't returned by then.
#
# general_kb = build_general_kb(n, box_h, box_w, givens)
# r, c, v = 1, 1, givens.get((1, 1), 1)  # try one cell/value pair
# query = atom('Is', r, c, v)
# 
# print(pl_resolution(general_kb, query))    
# print(tt_entails(associate('&', general_kb.clauses), query)) 

### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


Neither resolution nor model-checking finished within about 30 seconds, so I interrupted both executions.

For resolution, the main problem is the rapid growth in the number of clauses. In each iteration, the algorithm considers pairs of clauses and generates new resolvents. The newly generated resolvents are then added to the clause set, producing even more possible pairs in later rounds. Therefore, both the running time and memory usage can grow combinatorially.

For model-checking, the problem is the size of the truth-table state space. With m propositional symbols, truth-table model checking may need to consider up to 2^m possible truth assignments. A 9×9 Sudoku encoding contains hundreds of propositional atoms, so the number of possible models becomes enormous. Repeating this process for many cell-value queries makes solving the full grid computationally impractical.

**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [ ]:
# Implement solve_full_grid_fc(), pl_bc_entails(), and solve_full_grid_bc()
# in sudoku_solver.py. Rerun the import cell near the top after making changes.


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [3]:
# Verify solve_full_grid_fc against the puzzle's known solution. Then check
# pl_bc_entails directly, for completeness (it finds the correct value) and
# soundness (it never wrongly confirms an incorrect one), before timing
# solve_full_grid_bc against the same solution.
#
import time

t0 = time.time()
solved = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.time() - t0
assert solved == puzzle['solution']

definite_kb = build_definite_kb(n, box_h, box_w, givens)

# # Completeness: pl_bc_entails must find the correct value for every cell.
for (r, c), v in puzzle['solution'].items():
    assert pl_bc_entails(definite_kb, atom('Is', r, c, v)) == True

# # Soundness: pl_bc_entails must not also confirm any incorrect value.
for (r, c), v in puzzle['solution'].items():
    for other_v in range(1, n + 1):
        if other_v != v:
            assert pl_bc_entails(definite_kb, atom('Is', r, c, other_v)) == False

t0 = time.time()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.time() - t0
assert solved_bc == puzzle['solution']

print(f"solve_full_grid_fc: {fc_time:.2f}s")
print(f"solve_full_grid_bc: {bc_time:.2f}s")

solve_full_grid_fc: 19.35s
solve_full_grid_bc: 1.17s


In [4]:
results = []

for i, puzzle in enumerate(puzzle_pool, start=1):
    givens = puzzle['givens']

    t0 = time.time()
    solved_fc = solve_full_grid_fc(n, box_h, box_w, givens)
    fc_time = time.time() - t0

    t0 = time.time()
    solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
    bc_time = time.time() - t0

    assert solved_fc == puzzle['solution']
    assert solved_bc == puzzle['solution']

    results.append((i, fc_time, bc_time))

for i, fc_time, bc_time in results:
    print(f"Puzzle {i}: FC = {fc_time:.3f}s, BC = {bc_time:.3f}s")

Puzzle 1: FC = 19.639s, BC = 1.159s
Puzzle 2: FC = 19.756s, BC = 1.186s
Puzzle 3: FC = 19.780s, BC = 1.183s
Puzzle 4: FC = 19.666s, BC = 1.183s
Puzzle 5: FC = 19.693s, BC = 1.190s


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Your answer:**

Let I[r,c,v] mean that cell (r,c) has value v, and N[r,c,v] mean that the cell cannot have value v.

(a) General CNF knowledge base

1. At least one value per cell: I[r,c,1] OR ... OR I[r,c,n].
2. At most one value per cell: for v < w, add ~I[r,c,v] OR ~I[r,c,w].
3. No repeated values within a row, column or box: for distinct peer cells a,b and each value v, add ~I[a,v] OR ~I[b,v].
4. For each given (r,c)=v, add the fact I[r,c,v].

Rows, columns and boxes are handled uniformly through _peers, which returns a deduplicated collection of peer cells. When constructing the general KB, constraints are added only for cell pairs satisfying (r,c) < (rr,cc), avoiding symmetric duplicates. Every formula passed to PropKB.tell is already a CNF clause.

There is no need to add a separate constraint requiring each digit to appear at least once in each row: a row has n cells, each taking exactly one value from 1..n, and repeated values are prohibited. Its n distinct values must therefore cover all digits. The same reasoning applies to columns and boxes.

Each cell in a standard 9x9 grid has 20 peers. For the first puzzle, with 30 givens, the clause count is:

81 at-least-one clauses + 81*C(9,2)=2916 within-cell exclusion clauses + (81*20/2)*9=7290 peer exclusion clauses + 30 facts = 10317 clauses.

The general KB needs only 729 Is atoms and uses the logical operator ~ directly for negation.

(b) Definite-clause knowledge base

A definite clause has exactly one positive literal; a Horn clause more generally allows at most one. PropDefiniteKB accepts either a positive fact or an implication of the form 'a conjunction of positive atoms ==> one positive atom'. Consequently, the at-least-one disjunction containing n positive literals cannot be inserted directly.

This implementation introduces independent positive Not atoms and encodes sound Sudoku elimination rules:

- Given facts: I[r,c,v].
- Within-cell elimination: I[r,c,v] ==> N[r,c,w], where w != v.
- Peer elimination: I[a,v] ==> N[b,v], where a and b share a row, column or box.
- Last candidate: AND(N[r,c,w] for w != v) ==> I[r,c,v].

For example, I[1,2,3] implies N[2,2,3]. A cell's sole remaining value can be inferred only after all eight other values have been explicitly proved eliminated. Failure to prove Is cannot itself justify deriving Not. On a 1x1 board, the last-candidate rule has no premises, so the sole fact Is1_1_1 is added directly.

The first puzzle's Horn KB contains:

81*9*8=5832 within-cell elimination rules + 81*9*20=14580 peer elimination rules + 81*9=729 last-candidate rules + 30 givens = 21171 rules/facts.

It uses 1458 distinct propositional atoms across the Is and Not families.

Expressive limitations: Not is the name of a positive atom and does not automatically mean ~Is. The last-candidate rule is a sound inference rule derived from Sudoku constraints, not an equivalent rewriting of the original positive disjunction. Assigning all Is and Not atoms true would satisfy this purely definite-clause theory, so it cannot fully enforce Sudoku's mutual-exclusion semantics. It is a collection of sound inference rules for valid Sudoku puzzles, rather than a logically equivalent transformation of the full CNF. Complete inference over the Horn KB does not mean that every Sudoku can be solved. The supplied puzzles can be solved using these basic rules; when the rules are insufficient, B's solver should report unresolved cells.

Indexing: _IndexedDefiniteKB extends the supplied PropDefiniteKB without modifying the supporting files. by_premise lets FC quickly find affected rules, while by_head provides candidate rules for a BC goal. When the original FC algorithm processes an atom, clauses_with_premise records it in fc_seen, allowing B to retrieve all conclusions after a complete closure computation. Adding or removing knowledge clears inference state to prevent stale cached results. Input validation checks local conflicts; it does not establish in advance that the puzzle has a unique solution.


### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Your answer:**

I would not use either model checking or resolution-refutation as the default algorithm for solving Sudoku puzzles. Although both methods are sound and complete, these properties guarantee correctness and eventual termination, not computational efficiency.

Model checking has exponential state-space growth. With m propositional symbols, it may need to examine up to 2^m truth assignments. Since a Sudoku encoding contains hundreds of propositional atoms, exhaustive model checking becomes infeasible very quickly.

Resolution-refutation avoids explicitly enumerating all truth assignments, but it can still suffer from a combinatorial explosion in the number of clauses. It repeatedly examines pairs of clauses and adds newly generated resolvents. As the clause set grows, the number of possible clause pairs and intermediate resolvents can become extremely large, increasing both execution time and memory consumption.

This was also reflected in our observations: neither resolution nor model-checking returned within approximately 30 seconds for the attempted query on the full general Sudoku KB, so both executions had to be interrupted. Since solving the whole Sudoku would require answering many such cell-value queries, using either method as the default solver would be impractical. More targeted inference methods such as forward or backward chaining over the definite-clause representation are therefore more suitable for this assignment.



### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Your answer:**

**3.1 pseudocode**

```
function PL-BC-ENTAILS(KB, query) returns true or false
    inputs:
        KB: a knowledge base of propositional definite clauses
        query: a propositional symbol

    obtain valid cached state, or initialize:
        known ← facts in KB
        rules_by_head ← rules indexed by conclusion
        proofs ← proofs of initial facts

    if query is in known:
        return true

    active ← empty set
    failed_this_pass ← empty set

    function PROVE(goal) returns true or false
        if goal is in known:
            return true

        if goal is in active or failed_this_pass:
            return false

        add goal to active

        for each rule c in rules_by_head[goal]:
            count ← number of distinct premises in c

            for each premise p in c:
                if PROVE(p):
                    count ← count - 1
                else:
                    break

            if count = 0:
                add goal to known
                record c's premises as the proof of goal
                remove goal from active
                return true

        add goal to failed_this_pass
        remove goal from active
        return false

    repeat:
        previous_count ← size(known)
        clear failed_this_pass

        if PROVE(query):
            return true

        if size(known) = previous_count:
            return false
```

**3.2 Discussion**

The main challenges were avoiding cycles and handling failed proofs correctly. Search starts from the query and recursively proves the premises of matching rules. active blocks repeated goals on the current recursion path. Failures are cached only within a pass and reconsidered if new conclusions are proved. A rule succeeds only when all its premises are proved; alternative rules provide other possible proofs. Each pass terminates because cycles and repeated failed searches are blocked. Another pass is attempted only after the known set grows, which can happen only finitely often in a finite KB. Cached proofs are invalidated when the KB changes.


### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Your answer:**

Let U be a row, column or box; let a and b be two distinct cells in U; and let x and y be two distinct values. Define the premise P as:

```text
AND(N[a,v] for v not in {x,y})
AND
AND(N[b,v] for v not in {x,y})
```

This represents explicit evidence that both cells can take values only from {x,y}.

For every other cell u in U - {a,b}, add two separate rules:

```text
P ==> N[u,x]
P ==> N[u,y]
```

Each rule has positive Not atoms as premises and a single positive atom as its conclusion, so it is a definite clause. The two conclusions must be expressed as two separate rules. This encoding neither requires a negative condition such as 'an elimination fact has not yet been proved' nor treats the absence of a proof as evidence that a candidate is available.

Soundness: in a valid Sudoku, a and b each have one value and cannot repeat a value within the same unit. Since both must take values from {x,y}, together they must occupy x and y. Other cells in the unit cannot take either value. This elimination remains valid even if one of the two cells has already been resolved.

For example, if two cells in a row have eliminated every digit except 2 and 7, the remaining seven cells in that row can eliminate 2 and 7. These new elimination facts may then trigger further last-candidate rules.

Cost: a 9x9 Sudoku has 27 units. Each unit has C(9,2)=36 cell pairs and 36 value pairs. Each configuration adds two rules for each of the other seven cells. Naive enumeration therefore adds:

27 * 36 * 36 * 7 * 2 = 489888 rules, before deduplication.

Each rule has 2*(9-2)=14 premises, increasing KB construction, indexing, storage and inference costs. Advanced rules can expand the range of solvable puzzles but do not guarantee that every Sudoku can be solved.

The implementation follows the assignment's basic requirements by implementing elimination and last-candidate reasoning. The Naked Pairs rules above are a proposed encoding for this conceptual question; the nearly 490,000 additional rules have not been added to build_definite_kb.


### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**Your answer:**

(a) Backward chaining can be much faster when a large KB has many unrelated rule components and the query depends on only a small component. It starts with the query, finds rules that could prove it, and recursively explores their premises. Forward chaining may process many irrelevant consequences before reaching the requested conclusion. Although the first BC query takes time to scan the KB and build its index, later queries can reuse that index and previously proved facts.

(b) BC may offer little advantage when a query involves most of the KB, many candidate rules fail, or multiple queries collectively require most of the conclusions. Computing all entailed conclusions once using indexed forward chaining may be faster. Our recursive implementation may also need multiple passes to reconsider temporary failures after new conclusions are proved. These passes add work in cyclic KBs. Successful proofs are cached, but this does not eliminate all repeated search. Therefore, neither method is universally faster.

| Puzzle | FC (seconds) | Recursive BC (seconds) |
| --- | ---: | ---: |
| 1 | 7.907 | 0.725 |
| 2 | 8.310 | 0.723 |
| 3 | 8.325 | 0.729 |
| 4 | 8.494 | 0.736 |
| 5 | 8.553 | 0.729 |

Each solver was timed once per puzzle on the same machine, including KB construction. BC was faster on all five puzzles. In our implementation, FC restarts inference for each candidate query, while BC reuses its index and proved conclusions within the same KB. This helps explain the timing difference but does not mean BC is always faster than FC.


## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

Paste your deployed Streamlit app URL here:

`https://...`
